# 🎬 ASTRAA — LTX-Video Colab
Free/open-source image-to-video test for ASTRAA.

## 1. Check GPU
Run this first. A free T4 is ideal for this lightweight test.

In [ ]:
!nvidia-smi


## 2. Install LTX-Video
Uses the official Lightricks repository.

In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone --depth 1 https://github.com/Lightricks/LTX-Video.git
%cd /content/LTX-Video
!pip install -q -e '.[inference]'
!pip install -q --force-reinstall --no-deps 'huggingface-hub~=0.30'


## 3. Download the lighter 2B distilled model
This is intended for lighter VRAM than the 13B model.

In [ ]:
from huggingface_hub import hf_hub_download
model_dir='/content/LTX-Video/models'
import os
os.makedirs(model_dir, exist_ok=True)
hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=model_dir)
print('Model downloaded:', model_dir)


## 4. Upload the ASTRAA reference image
Upload one image such as Aarav + Maa Meera. Keep the image in `/content/LTX-Video/`.

In [ ]:
from google.colab import files
uploaded=files.upload()
image_name=next(iter(uploaded))
image_path=f'/content/LTX-Video/{image_name}'
print(image_path)


## 5. Generate the first test shot
Start with a short 49-frame test first. The current LTX-Video 0.9.8 config supplies the checkpoint itself, so we do not pass `--checkpoint_path` separately.


In [ ]:
import os, subprocess, yaml
from huggingface_hub import hf_hub_download

base='/content/LTX-Video'
os.makedirs(f'{base}/models', exist_ok=True)
os.makedirs(f'{base}/configs', exist_ok=True)

# Make sure the official inference script and 2B config exist.
for name in ['inference.py', 'configs/ltxv-2b-0.9.8-distilled.yaml']:
    path=f'{base}/{name}'
    if not os.path.isfile(path):
        url=f'https://raw.githubusercontent.com/Lightricks/LTX-Video/main/{name}'
        subprocess.run(['wget','-q','-O',path,url], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'LTX inference failed with exit code {result.returncode}')

    raise RuntimeError(f'LTX inference failed with exit code {result.returncode}')

model=f'{base}/models/ltxv-2b-0.9.8-distilled.safetensors'
if not os.path.isfile(model):
    hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-2b-0.9.8-distilled.safetensors', local_dir=f'{base}/models')

upscaler=f'{base}/models/ltxv-spatial-upscaler-0.9.8.safetensors'
if not os.path.isfile(upscaler):
    hf_hub_download(repo_id='Lightricks/LTX-Video', filename='ltxv-spatial-upscaler-0.9.8.safetensors', local_dir=f'{base}/models')

# Local config: use downloaded models and disable prompt enhancement for lower T4 memory use.
with open(f'{base}/configs/ltxv-2b-0.9.8-distilled.yaml') as f:
    cfg=yaml.safe_load(f)
cfg['checkpoint_path']=model
cfg['spatial_upscaler_model_path']=upscaler
cfg['prompt_enhancement_words_threshold']=0
local_cfg=f'{base}/configs/astrraa-2b-t4.yaml'
with open(local_cfg,'w') as f:
    yaml.safe_dump(cfg,f,sort_keys=False)

image_candidates = []
for ext in ('*.jpg','*.jpeg','*.png','*.webp'):
    image_candidates.extend(__import__('glob').glob(f'{base}/{ext}'))
if 'image_path' not in globals() or not os.path.isfile(image_path):
    if not image_candidates:
        raise FileNotFoundError('No reference image found in /content/LTX-Video. Run Step 4 once to upload it.')
    image_path = image_candidates[0]
print('Using reference image:', image_path)

PROMPT='Aarav and his mother remain visually consistent with the reference image. They are inside their house at night. The curtains gently move in the wind. Aarav slowly looks toward the entrance with a worried expression. His mother moves slightly closer to protect him. Slow cinematic camera push-in, subtle dust particles, dramatic nighttime lighting, high-quality 3D animated fantasy movie style, natural motion, no dialogue, no text.'
print('READY:', os.path.isfile(f'{base}/inference.py'), os.path.isfile(local_cfg), os.path.isfile(model), os.path.isfile(upscaler))

subprocess.run([
    'python', f'{base}/inference.py',
    '--prompt', PROMPT,
    '--conditioning_media_paths', image_path,
    '--conditioning_start_frames', '0',
    '--height', '512', '--width', '768',
    '--num_frames', '49', '--seed', '42',
    '--pipeline_config', local_cfg,
], capture_output=True, text=True)





In [ ]:
import os, glob
videos=glob.glob('/content/LTX-Video/**/*.mp4', recursive=True)
print('\n'.join(videos[-10:]) if videos else 'No MP4 found yet.')


## Next
Once the first shot works, we will add reusable ASTRAA prompts and an extension workflow.